In [4]:
import sys
sys.path.insert(0, '/Users/sihamchowdhury/Desktop/Projects/finance-dashboards')


# Exploring Labour, Pipeline & Targets Tables

Three tables not yet used in the dashboard:
- `core.fact_sd_timesheet_cost` — people costs by role
- `core.rpt_hubspot_line_report` — pipeline deals
- `core.fact_hubspot_targets` — quarterly revenue targets

Goal: understand grain, quality and how to best use each in the dashboard.

In [23]:
import pandas as pd
from src.db.connection import get_engine

engine = get_engine()
engine.dispose()
print("Connected")

Connected


---
## 1. Labour — `fact_sd_timesheet_cost`

Daily timesheet entries from Screendragon. Each row is one person × one project × one day.

The SQL classifies roles into three labour types:
- **Research Labor** — VP, SVP, Director, Manager roles
- **Ops Labor** — Field, QA, Programming, Ops, Dashboard roles  
- **Other Labor** — everything else

In [24]:
# Total labour by type and year
df_labour = pd.read_sql("""
    SELECT
        EXTRACT(YEAR FROM accounting_period_start_date)::int AS yr,
        CASE
            WHEN role_name_at_timesheet_date ILIKE ANY(ARRAY['%%VP%%','%%SVP%%','%%Director%%','%%Manager%%'])
            THEN 'Research Labor'
            WHEN role_name_at_timesheet_date ILIKE ANY(ARRAY['%%Field%%','%%QA%%','%%Programming%%','%%Ops%%','%%Dashboard%%'])
            THEN 'Ops Labor'
            ELSE 'Other Labor'
        END AS labor_type,
        SUM(timesheet_external_cost) AS labor_cost
    FROM core.fact_sd_timesheet_cost
    WHERE accounting_period_is_posted = TRUE
    GROUP BY 1, 2
    ORDER BY 1, 3 DESC
""", engine)

print(df_labour.pivot(index='yr', columns='labor_type', values='labor_cost').round(0))

labor_type   Ops Labor  Other Labor  Research Labor
yr                                                 
2024        15093886.0   18399093.0      41713392.0
2025        36409531.0   20416496.0      43330018.0
2026        10041278.0    2955289.0       8306608.0


In [25]:
# Total labour by type and year
df_labour = pd.read_sql("""
    SELECT *
    FROM core.fact_sd_timesheet_cost
    WHERE accounting_period_is_posted = TRUE
""", engine)

df_labour.head()

,netsuite_project_id,sd_project_id,netsuite_project_number,user_id,category,actual_hours,timesheet_date,user_first_name,user_last_name,role_name_at_timesheet_date,...,external_rate_at_timesheet_date,timesheet_external_cost,internal_rate_at_timesheet_date,timesheet_internal_cost,accounting_period_id,accounting_period_start_date,accounting_period_ending_date,accounting_period_closed_at,accounting_period_is_posted,accounting_period_is_closed
0,133238.0,PRJ53001,PRJ53001,1061,Billable - Operations,0.0,2024-11-25,A,Jatin,Ops - Field Management (MK),...,137.0,0.0,47.0,0.0,135,2024-11-01,2024-11-30,2026-01-28 10:23:51+00:00,True,True
1,133238.0,PRJ53001,PRJ53001,1061,Billable - Operations,0.0,2024-11-26,A,Jatin,Ops - Field Management (MK),...,137.0,0.0,47.0,0.0,135,2024-11-01,2024-11-30,2026-01-28 10:23:51+00:00,True,True
2,133238.0,PRJ53001,PRJ53001,1061,Billable - Operations,0.0,2024-11-27,A,Jatin,Ops - Field Management (MK),...,137.0,0.0,47.0,0.0,135,2024-11-01,2024-11-30,2026-01-28 10:23:51+00:00,True,True
3,133238.0,PRJ53001,PRJ53001,1061,Billable - Operations,0.0,2024-11-28,A,Jatin,Ops - Field Management (MK),...,137.0,0.0,47.0,0.0,135,2024-11-01,2024-11-30,2026-01-28 10:23:51+00:00,True,True
4,133238.0,PRJ53001,PRJ53001,1061,Billable - Operations,0.0,2024-11-29,A,Jatin,Ops - Field Management (MK),...,137.0,0.0,47.0,0.0,135,2024-11-01,2024-11-30,2026-01-28 10:23:51+00:00,True,True


In [26]:
# Monthly labour by type for 2025
df_labour_monthly = pd.read_sql("""
    SELECT
        accounting_period_start_date,
        CASE
            WHEN role_name_at_timesheet_date ILIKE ANY(ARRAY['%%VP%%','%%SVP%%','%%Director%%','%%Manager%%'])
            THEN 'Research Labor'
            WHEN role_name_at_timesheet_date ILIKE ANY(ARRAY['%%Field%%','%%QA%%','%%Programming%%','%%Ops%%','%%Dashboard%%'])
            THEN 'Ops Labor'
            ELSE 'Other Labor'
        END AS labor_type,
        SUM(timesheet_external_cost) AS labor_cost,
        SUM(actual_hours) AS total_hours
    FROM core.fact_sd_timesheet_cost
    WHERE accounting_period_is_posted = TRUE
      AND EXTRACT(YEAR FROM accounting_period_start_date) = 2025
    GROUP BY 1, 2
    ORDER BY 1, 3 DESC
""", engine)

df_labour_monthly["accounting_period_start_date"] = pd.to_datetime(df_labour_monthly["accounting_period_start_date"])
print(f"Rows: {len(df_labour_monthly)}")
df_labour_monthly.head(10)

Rows: 36


,accounting_period_start_date,labor_type,labor_cost,total_hours
0,2025-01-01,Research Labor,3980745.30,17059.05
1,2025-01-01,Ops Labor,3078910.60,22473.80
2,2025-01-01,Other Labor,1950305.85,16454.85
3,2025-02-01,Research Labor,3568271.10,15291.25
4,2025-02-01,Ops Labor,3063607.70,22362.10
5,2025-02-01,Other Labor,1791658.90,14159.70
6,2025-03-01,Research Labor,3598288.50,15545.95
7,2025-03-01,Ops Labor,3226870.60,23553.80
8,2025-03-01,Other Labor,1882262.20,13316.95
9,2025-04-01,Research Labor,3727760.10,16209.15


In [27]:
# Top roles by cost — useful to validate the classification
pd.read_sql("""
    SELECT
        role_name_at_timesheet_date,
        COUNT(*) as rows,
        SUM(actual_hours) as total_hours,
        SUM(timesheet_external_cost) as total_cost
    FROM core.fact_sd_timesheet_cost
    WHERE accounting_period_is_posted = TRUE
      AND EXTRACT(YEAR FROM accounting_period_start_date) = 2025
    GROUP BY 1
    ORDER BY 4 DESC
    LIMIT 20
""", engine)

,role_name_at_timesheet_date,rows,total_hours,total_cost
0,NaN,7942,4877.00,NaN
1,Research Director,99647,69133.00,16799319.00
2,Research Manager,104101,87945.40,15302499.60
3,Research Analyst,93997,85757.15,11748729.55
4,Engineer,15344,49814.75,8667766.50
5,Research VP,62324,24845.90,8124609.30
6,Ops - Data Processing (MK),28023,24436.30,3347773.10
7,Ops - Ad Editorial,6983,23069.80,3160562.60
8,Research SVP,16742,5911.60,3103590.00
9,Ops - Programming (DM),27201,18725.40,2565379.80


In [28]:
# Simplified project count check
pd.read_sql("""
    SELECT COUNT(DISTINCT netsuite_project_id) AS timesheet_projects_2025
    FROM core.fact_sd_timesheet_cost
    WHERE EXTRACT(YEAR FROM accounting_period_start_date) = 2025
""", engine)


,timesheet_projects_2025
0,1168


In [29]:
# 2025 labour summary — total cost, hours, cost per hour by type
df_labour_summary = pd.read_sql("""
    SELECT
        CASE
            WHEN role_name_at_timesheet_date ILIKE ANY(ARRAY['%%VP%%','%%SVP%%','%%Director%%','%%Manager%%'])
            THEN 'Research Labor'
            WHEN role_name_at_timesheet_date ILIKE ANY(ARRAY['%%Field%%','%%QA%%','%%Programming%%','%%Ops%%','%%Dashboard%%'])
            THEN 'Ops Labor'
            ELSE 'Other Labor'
        END AS labor_type,
        SUM(actual_hours) AS total_hours,
        SUM(timesheet_external_cost) AS total_cost
    FROM core.fact_sd_timesheet_cost
    WHERE accounting_period_is_posted = TRUE
      AND EXTRACT(YEAR FROM accounting_period_start_date) = 2025
    GROUP BY 1
    ORDER BY 3 DESC
""", engine)

df_labour_summary["cost_per_hour"] = (df_labour_summary["total_cost"] / df_labour_summary["total_hours"]).round(2)
print(df_labour_summary.round(0))

       labor_type  total_hours  total_cost  cost_per_hour
0  Research Labor     187836.0  43330018.0          231.0
1       Ops Labor     265763.0  36409531.0          137.0
2     Other Labor     145817.0  20416496.0          140.0


---
## 2. Pipeline — `rpt_hubspot_line_report`

HubSpot deal lines — active pipeline (Closed Won and Closed Lost excluded).

Key questions:
- What stages exist and how much value is at each stage?
- How does pipeline break down by service line and vertical?
- Who are the top owners by pipeline value?

In [30]:
# Pipeline by stage
df_pipe = pd.read_sql("""
    SELECT
        deal_id,
        deal_pipeline_stage_name,
        COALESCE(vertical, 'Unassigned') AS vertical,
        COALESCE(service_line, 'Unassigned') AS service_line,
        owner_full_name,
        deal_amount_usd AS pipeline_value_usd
    FROM core.rpt_hubspot_line_report
    WHERE is_deal_deleted = FALSE
      AND LOWER(deal_pipeline_stage_name) NOT IN ('closed won', 'closed lost')
      AND deal_amount_usd IS NOT NULL
""", engine)

print(f"Total pipeline rows: {len(df_pipe)}")
print(f"Unique deals: {df_pipe['deal_id'].nunique()}")
print(f"Total pipeline value: ${df_pipe.drop_duplicates('deal_id')['pipeline_value_usd'].sum()/1e6:.1f}M")

print("\nBy stage:")
stage_summary = (
    df_pipe.drop_duplicates("deal_id")
    .groupby("deal_pipeline_stage_name")
    .agg(deals=("deal_id","nunique"), value=("pipeline_value_usd","sum"))
    .sort_values("value", ascending=False)
)
print(stage_summary.round(0))

Total pipeline rows: 1314
Unique deals: 1118
Total pipeline value: $328.2M

By stage:
                                 deals       value
deal_pipeline_stage_name                          
Target Account                      57  91361500.0
0% Prospecting Pipeline            635  46493037.0
Q1 2025                             24  38550000.0
Q2 2025                             23  38400000.0
Q3 2025                             22  32000000.0
Q4 2025                             23  32000000.0
Stage 2 - Solutioning               66  14263892.0
Stage 1 - Qualifying                76  12382121.0
Stage 3 - Proposed                  65   9029757.0
Stage 5 - Contracting               24   4093017.0
Stage 4 - Negotiating / Closing     27   3370934.0
0% Prospecting                      39   2981666.0
Stage 6 - Pending                   20   1844310.0
0% Capabilities                      7   1042000.0
Stage 0 - Prospecting               10    425000.0


In [31]:
# Pipeline by service line
print("By service line:")
sl_summary = (
    df_pipe.drop_duplicates("deal_id")
    .groupby("service_line")
    .agg(deals=("deal_id","nunique"), value=("pipeline_value_usd","sum"))
    .sort_values("value", ascending=False)
)
print(sl_summary.round(0))

By service line:
                                                   deals        value
service_line                                                         
Unassigned                                           316  151116666.0
Campaign Analytics                                   283   42671651.0
Brand Tracking                                       106   25101769.0
Custom Research                                      291   18471979.0
Campaign Analytics;Brand Tracking;Custom Research      7   12800000.0
Campaign Analytics;Brand Tracking;Content              1   11500000.0
Campaign Analytics;Custom Research                    19    9973000.0
Brand Tracking;Campaign Analytics;Custom Research      2    9100000.0
Brand Tracking;Data Science;Campaign Analytics         1    8800000.0
Campaign Analytics;Brand Tracking                      9    6619000.0
Data Science                                          14    4316234.0
Content                                               27    3617535.0
Dat

In [32]:
# Top 10 owners by pipeline value
print("Top 10 owners:")
owner_summary = (
    df_pipe.drop_duplicates("deal_id")
    .groupby("owner_full_name")
    .agg(deals=("deal_id","nunique"), value=("pipeline_value_usd","sum"))
    .sort_values("value", ascending=False)
    .head(10)
)
print(owner_summary.round(0))

Top 10 owners:
                    deals       value
owner_full_name                      
Lena Kasahara          40  31813750.0
Mara Doran             76  30774000.0
Danielle Byrd          31  28350000.0
Christian Dubreuil     67  27143750.0
Dounia Turrill         83  25153710.0
Mike Chung             33  24315000.0
Greg Conlon            97  23400573.0
Mark Willard           41  23043600.0
Jon Neary              35  16563740.0
Ben Carlson             5  13025000.0


In [33]:
# Average deal size by stage
print("Average deal size by stage:")
avg_deal = (
    df_pipe.drop_duplicates("deal_id")
    .groupby("deal_pipeline_stage_name")
    .agg(deals=("deal_id","nunique"), value=("pipeline_value_usd","sum"))
    .reset_index()
)
avg_deal["avg_deal_size"] = (avg_deal["value"] / avg_deal["deals"]).round(0)
print(avg_deal.sort_values("value", ascending=False).to_string(index=False))

Average deal size by stage:
       deal_pipeline_stage_name  deals        value  avg_deal_size
                 Target Account     57 9.136150e+07      1602833.0
        0% Prospecting Pipeline    635 4.649304e+07        73217.0
                        Q1 2025     24 3.855000e+07      1606250.0
                        Q2 2025     23 3.840000e+07      1669565.0
                        Q3 2025     22 3.200000e+07      1454545.0
                        Q4 2025     23 3.200000e+07      1391304.0
          Stage 2 - Solutioning     66 1.426389e+07       216120.0
           Stage 1 - Qualifying     76 1.238212e+07       162923.0
             Stage 3 - Proposed     65 9.029757e+06       138919.0
          Stage 5 - Contracting     24 4.093017e+06       170542.0
Stage 4 - Negotiating / Closing     27 3.370934e+06       124849.0
                 0% Prospecting     39 2.981666e+06        76453.0
              Stage 6 - Pending     20 1.844310e+06        92216.0
                0% Capabilities   

---
## 3. Targets — `fact_hubspot_targets`

Quarterly revenue targets by person and team. Only available from Q1 2025 onwards.

Key questions:
- What teams and people have targets?
- Can we compare targets to actual pipeline won?
- What's the team-level target for 2025?

In [34]:
# Targets overview
df_targets = pd.read_sql("""
    SELECT
        quarter_start_date,
        quarter_end_date,
        team_primary_name,
        user_first_name || ' ' || user_last_name AS person,
        target_amount_usd
    FROM core.fact_hubspot_targets
    ORDER BY quarter_start_date, team_primary_name
""", engine)

df_targets["quarter_start_date"] = pd.to_datetime(df_targets["quarter_start_date"])
print(f"Total target rows: {len(df_targets)}")
print(f"Date range: {df_targets['quarter_start_date'].min()} to {df_targets['quarter_start_date'].max()}")
print(f"Teams: {df_targets['team_primary_name'].unique().tolist()}")
df_targets.head(10)

Total target rows: 76
Date range: 2025-01-01 00:00:00 to 2025-10-01 00:00:00
Teams: ['Brands - CPG', 'Europe', 'FSA', 'MSE', 'Tech - Telco']


,quarter_start_date,quarter_end_date,team_primary_name,person,target_amount_usd
0,2025-01-01,2025-03-31,Brands - CPG,Lena Kasahara,3500000
1,2025-01-01,2025-03-31,Brands - CPG,Meghan Teed,0
2,2025-01-01,2025-03-31,Brands - CPG,Joanna Fanuele,2200000
3,2025-01-01,2025-03-31,Brands - CPG,Juliann Pavlovcic,2700000
4,2025-01-01,2025-03-31,Brands - CPG,Duncan Stark,1500000
5,2025-01-01,2025-03-31,Europe,Christian Dubreuil,1800000
6,2025-01-01,2025-03-31,FSA,Jon Neary,5800000
7,2025-01-01,2025-03-31,FSA,Mike Chung,1500000
8,2025-01-01,2025-03-31,FSA,Mark Willard,5600000
9,2025-01-01,2025-03-31,FSA,Mark Cosby,0


In [35]:
# Annual target by team for 2025
targets_2025 = df_targets[df_targets["quarter_start_date"].dt.year == 2025]

print("2025 targets by team:")
print(
    targets_2025.groupby("team_primary_name")["target_amount_usd"]
    .sum()
    .sort_values(ascending=False)
    .round(0)
)

print(f"\nTotal 2025 target: ${targets_2025['target_amount_usd'].sum()/1e6:.1f}M")

2025 targets by team:
team_primary_name
Tech - Telco    35600000
FSA             34600000
MSE             33550000
Brands - CPG    29100000
Europe           8100000
Name: target_amount_usd, dtype: int64

Total 2025 target: $140.9M


In [36]:
# Quarterly target totals
print("Quarterly targets (all teams):")
print(
    df_targets.groupby("quarter_start_date")["target_amount_usd"]
    .sum()
    .round(0)
)

Quarterly targets (all teams):
quarter_start_date
2025-01-01    38550000
2025-04-01    38400000
2025-07-01    32000000
2025-10-01    32000000
Name: target_amount_usd, dtype: int64


In [37]:
# How does closed won pipeline compare to targets?
# Closed won deals from HubSpot
df_closed_won = pd.read_sql("""
    SELECT
        DATE_TRUNC('quarter', closed_date::timestamp)::date AS quarter_start,
        COUNT(DISTINCT deal_id) AS deals_won,
        SUM(deal_amount_usd) AS revenue_won
    FROM core.rpt_hubspot_line_report
    WHERE is_deal_deleted = FALSE
      AND LOWER(deal_pipeline_stage_name) = 'closed won'
      AND deal_amount_usd IS NOT NULL
      AND closed_date IS NOT NULL
      AND EXTRACT(YEAR FROM closed_date::timestamp) = 2025
    GROUP BY 1
    ORDER BY 1
""", engine)

print("2025 Closed Won by quarter:")
print(df_closed_won)

2025 Closed Won by quarter:
  quarter_start  deals_won   revenue_won
0    2025-01-01        275  4.675980e+07
1    2025-04-01        282  5.322861e+07
2    2025-07-01        244  2.554115e+07
3    2025-10-01        205  1.431646e+08


In [38]:
# Monthly labour cost trend 2024 vs 2025
df_labour_trend = pd.read_sql("""
    SELECT
        EXTRACT(YEAR FROM accounting_period_start_date)::int AS yr,
        EXTRACT(MONTH FROM accounting_period_start_date)::int AS month_num,
        CASE
            WHEN role_name_at_timesheet_date ILIKE ANY(ARRAY['%%VP%%','%%SVP%%','%%Director%%','%%Manager%%'])
            THEN 'Research Labor'
            WHEN role_name_at_timesheet_date ILIKE ANY(ARRAY['%%Field%%','%%QA%%','%%Programming%%','%%Ops%%','%%Dashboard%%'])
            THEN 'Ops Labor'
            ELSE 'Other Labor'
        END AS labor_type,
        SUM(timesheet_external_cost) AS labor_cost,
        SUM(actual_hours) AS total_hours
    FROM core.fact_sd_timesheet_cost
    WHERE accounting_period_is_posted = TRUE
      AND EXTRACT(YEAR FROM accounting_period_start_date) IN (2024, 2025)
    GROUP BY 1, 2, 3
    ORDER BY 1, 2, 3
""", engine)
print(f"Labour trend rows: {len(df_labour_trend)}")
print(df_labour_trend.groupby(["yr","labor_type"])["labor_cost"].sum().round(0))

Labour trend rows: 72
yr    labor_type    
2024  Ops Labor         15093886.0
      Other Labor       18399093.0
      Research Labor    41713392.0
2025  Ops Labor         36409531.0
      Other Labor       20416496.0
      Research Labor    43330018.0
Name: labor_cost, dtype: float64


---
## Summary — What to add to the dashboard

### Labour (`fact_sd_timesheet_cost`)
Best used as a **standalone labour section** in the dashboard:
- Stacked bar: monthly labour cost by type (Research / Ops / Other)
- Labour cost as % of revenue (labour intensity)
- YoY labour cost trend
- Hours vs cost split (efficiency)

### Pipeline (`rpt_hubspot_line_report`)
Best used as a **pipeline section**:
- Funnel chart by stage (total value at each stage)
- Pipeline by service line (bar)
- Top 10 owners by value (bar)
- Total pipeline value KPI card

### Targets (`fact_hubspot_targets`)
Best used for **actual vs target** tracking:
- Quarterly target vs actual revenue by team
- Attainment % per quarter
- Note: targets only from Q1 2025 — limited to 2025+ comparisons

In [39]:
import sys
sys.path.insert(0, '/Users/sihamchowdhury/Desktop/Projects/finance-dashboards')

import pandas as pd
from sqlalchemy import text
from src.db.connection import get_engine

engine = get_engine()

# test connection first
with engine.connect() as conn:
    conn.execute(text("SELECT 1"))
print("Connection ok")

Connection ok


In [40]:
from sqlalchemy import text

# 1. Labour by type and year
print("=== LABOUR BY TYPE & YEAR ===")
with engine.connect() as conn:
    df_lab = pd.read_sql(text("""
        SELECT
            EXTRACT(YEAR FROM accounting_period_start_date)::int AS yr,
            CASE
                WHEN role_name_at_timesheet_date ILIKE ANY(ARRAY['%VP%','%SVP%','%Director%','%Manager%'])
                THEN 'Research Labor'
                WHEN role_name_at_timesheet_date ILIKE ANY(ARRAY['%Field%','%QA%','%Programming%','%Ops%','%Dashboard%'])
                THEN 'Ops Labor'
                ELSE 'Other Labor'
            END AS labor_type,
            SUM(timesheet_external_cost) AS labor_cost,
            SUM(actual_hours) AS total_hours
        FROM core.fact_sd_timesheet_cost
        WHERE accounting_period_is_posted = TRUE
        GROUP BY 1, 2
        ORDER BY 1, 3 DESC
    """), conn)
print(df_lab.pivot(index='yr', columns='labor_type', values='labor_cost').round(0))

# 2. Pipeline by stage
print("\n=== PIPELINE BY STAGE ===")
with engine.connect() as conn:
    df_pipe = pd.read_sql(text("""
        SELECT
            deal_pipeline_stage_name,
            COUNT(DISTINCT deal_id) AS deals,
            SUM(deal_amount_usd) AS total_value
        FROM core.rpt_hubspot_line_report
        WHERE is_deal_deleted = FALSE
          AND LOWER(deal_pipeline_stage_name) NOT IN ('closed won', 'closed lost')
          AND deal_amount_usd IS NOT NULL
        GROUP BY 1
        ORDER BY 3 DESC
    """), conn)
print(df_pipe.to_string(index=False))

# 3. Targets by team and quarter
print("\n=== TARGETS BY TEAM & QUARTER ===")
with engine.connect() as conn:
    df_tgt = pd.read_sql(text("""
        SELECT
            quarter_start_date,
            team_primary_name,
            SUM(target_amount_usd) AS target
        FROM core.fact_hubspot_targets
        GROUP BY 1, 2
        ORDER BY 1, 3 DESC
    """), conn)
print(df_tgt.to_string(index=False))
print(f"\nTotal 2025 target: ${df_tgt[df_tgt['quarter_start_date'].astype(str).str.startswith('2025')]['target'].sum()/1e6:.1f}M")

=== LABOUR BY TYPE & YEAR ===
labor_type   Ops Labor  Other Labor  Research Labor
yr                                                 
2024        15093886.0   18399093.0      41713392.0
2025        36409531.0   20416496.0      43330018.0
2026        10041278.0    2955289.0       8306608.0

=== PIPELINE BY STAGE ===
       deal_pipeline_stage_name  deals  total_value
                 Target Account     57 9.136150e+07
        0% Prospecting Pipeline    635 5.724799e+07
                        Q1 2025     24 3.855000e+07
                        Q2 2025     23 3.840000e+07
                        Q4 2025     23 3.200000e+07
                        Q3 2025     22 3.200000e+07
          Stage 5 - Contracting     24 2.601261e+07
          Stage 2 - Solutioning     66 2.451072e+07
           Stage 1 - Qualifying     76 1.844306e+07
             Stage 3 - Proposed     65 1.827933e+07
Stage 4 - Negotiating / Closing     27 6.193534e+06
                 0% Prospecting     39 2.981666e+06
       